> 在线讲义：[第 38 章 · 位置编码的数学：正弦编码、RoPE、长度外推（PI、NTK、YaRN）、ALiBi](https://wikieden.github.io/math-and-eval-stats/lessons/38-position-encoding.html) · 本书《数学与实验统计》

# 第 38 章 · 位置编码的数学：正弦编码、RoPE、长度外推（PI、NTK、YaRN）、ALiBi

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/38-position-encoding.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 正弦平移 | R(kω) | `sinusoid_shift(10, 7)` | [0.5121, 0.8589] 两次 |
| 第二步 波长 | 2π/θ | `wavelengths(freqs())` | 6.28 … 54,410；18 对 > 4096 |
| 第三步 NTK | 换底数 | `ntk_base(4.0)` | 40,890 |
| 第三步 YaRN | 三段 | `yarn_regimes()` | 21 / 25 / 18，1.1386 |
| 第四步 ALiBi | 斜率 | `alibi_slopes()` | 1/2 … 1/256 |

例子：d = 128、底数 10000 的 RoPE；把上下文从 4096 扩到 16384（s = 4）时 PI、NTK-aware、YaRN 各怎么改频率；
8 个头的 ALiBi 斜率。
里程碑 ME-M24 的“数值验证 RoPE 只依赖相对位置”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
D, BASE, L_TRAIN = 128, 10000.0, 4096

**函数 `freqs`**：第 i 对维度的角频率 θ_i = base^{−2i/d}，i = 0 … d/2 − 1。

In [ ]:
def freqs(d: int = D, base: float = BASE) -> np.ndarray:
    """第 i 对维度的角频率 θ_i = base^{−2i/d}，i = 0 … d/2 − 1。"""
    return base ** (-np.arange(0, d, 2) / d)

**函数 `sinusoid_shift`**：正弦编码第 i 对 (sin pω, cos pω)：PE(p + k) = R(kω)·PE(p)，R 是只和 k 有关的 2×2 矩阵。返回 (直接算, 用矩阵算)。

In [ ]:
def sinusoid_shift(p: int, k: int, i: int = 3, d: int = 16) -> tuple[np.ndarray, np.ndarray]:
    """正弦编码第 i 对 (sin pω, cos pω)：PE(p + k) = R(kω)·PE(p)，R 是只和 k 有关的 2×2 矩阵。返回 (直接算, 用矩阵算)。"""
    w = 10000 ** (-2 * i / d)
    pe = lambda q: np.array([np.sin(q * w), np.cos(q * w)])  # noqa: E731
    r = np.array([[np.cos(k * w), np.sin(k * w)], [-np.sin(k * w), np.cos(k * w)]])
    return pe(p + k), r @ pe(p)

**函数 `wavelengths`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def wavelengths(th: np.ndarray) -> np.ndarray:
    return 2 * np.pi / th

**函数 `ntk_base`**：NTK-aware：把底数换成 base·s^{d/(d−2)}，最低频正好被压缩 s 倍，最高频几乎不动（YaRN 附录 A.2）。

In [ ]:
def ntk_base(s: float, d: int = D, base: float = BASE) -> float:
    """NTK-aware：把底数换成 base·s^{d/(d−2)}，最低频正好被压缩 s 倍，最高频几乎不动（YaRN 附录 A.2）。"""
    return base * s ** (d / (d - 2))

**函数 `yarn_regimes`**：YaRN 的 NTK-by-parts：r = L/λ 是训练长度里转的圈数。r > β 不插值，r < α 完全按 PI 插值，中间线性过渡。

In [ ]:
def yarn_regimes(s: float = 4.0, alpha: float = 1.0, beta: float = 32.0) -> tuple[int, int, int, float]:
    """YaRN 的 NTK-by-parts：r = L/λ 是训练长度里转的圈数。r > β 不插值，r < α 完全按 PI 插值，中间线性过渡。
    返回 (不插值的维度对数, 过渡的, 完全插值的, 注意力温度 √(1/t) = 0.1 ln s + 1)。"""
    r = L_TRAIN / wavelengths(freqs())
    return int((r > beta).sum()), int(((r >= alpha) & (r <= beta)).sum()), int((r < alpha).sum()), 0.1 * np.log(s) + 1

**函数 `alibi_slopes`**：ALiBi 第 k 个头的斜率 2^{−8k/h}，k = 1 … h；注意力分数加 −m·(i − j)。

In [ ]:
def alibi_slopes(h: int = 8) -> list[float]:
    """ALiBi 第 k 个头的斜率 2^{−8k/h}，k = 1 … h；注意力分数加 −m·(i − j)。"""
    return [2 ** (-8 * k / h) for k in range(1, h + 1)]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 正弦编码：平移 k 是一个只依赖 k 的旋转")
a, b = sinusoid_shift(10, 7)
print(f"  第 3 对维度：PE(17) = {np.round(a, 4).tolist()}，R(7ω)·PE(10) = {np.round(b, 4).tolist()}")

In [ ]:
print("\n第二步 RoPE：每对维度按位置转一个角度")
th = freqs()
lam = wavelengths(th)
print(f"  d = 128、底数 1e4：最快一对每个 token 转 {th[0]:.0f} 弧度（波长 {lam[0]:.2f}），最慢一对波长 {lam[-1]:,.0f}")
print(f"  波长超过训练长度 4096 的维度对：{int((lam > L_TRAIN).sum())} / 64")

In [ ]:
print("\n第三步 扩长度（4096 → 16384，s = 4）")
nb = ntk_base(4.0)
th2 = freqs(base=nb)
print(f"  位置插值 PI：所有频率除以 4（位置 m → m/4）")
print(f"  NTK-aware：底数 1e4 → {nb:,.0f}；最高频 {th[0]:.4f} → {th2[0]:.4f}（几乎不变），最低频缩小 {th[-1] / th2[-1]:.4f} 倍")
keep, ramp, interp, tmp = yarn_regimes()
print(f"  YaRN（α = 1、β = 32）：不插值 {keep} 对、过渡 {ramp} 对、完全插值 {interp} 对；注意力温度 √(1/t) = {tmp:.4f}")

In [ ]:
print("\n第四步 ALiBi：不加位置向量，直接在分数上减距离")
sl = alibi_slopes()
print(f"  8 个头的斜率：{[f'1/{round(1 / m)}' for m in sl]}")
print(f"  距离 100 时各头的偏置：{[round(-100 * m, 2) for m in sl]}")

## 3. 代码里才看得到的

1. **频率按 0, 2, 4, … 取下标**：`np.arange(0, d, 2)/d` 就是 2i/d，最后一对是 126/128，所以最慢波长是 2π·10000^{126/128} 而不是 2π·10000。
2. **YaRN 的边界用了 ≥ 和 ≤**：r 恰好等于 α 或 β 的维度算在“过渡”里，和论文斜坡函数在端点处的取值一致（γ = 0 或 1）。
3. **没有验证 RoPE 的相对性**：“同样的 m − n 得到同样的点积”留给自己做（里程碑 ME-M24）。

## 4. 自己改一改

1. 把底数改成 500000（Llama 3 的取值），重算第二步有多少对的波长超过 4096。
2. 把 s 改成 8，NTK-aware 的底数和 YaRN 三段各有几对？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 正弦编码是钟表，平移是旋转但加在输入上；
> RoPE 直接转 q 和 k，点积只剩 m 减 n；
> 慢钟没转满一圈，外推就出事；
> PI 全压缩，NTK 换底数，YaRN 分三段加温度；
> ALiBi 按距离减分，斜率成几何序列。